# NuggetKriging on the noisy Branin 2D function (Python)

This notebook demonstrates `NuggetKriging`, which estimates a **global nugget**
effect (homoscedastic noise variance) from the data. Unlike `NoiseKriging`
where noise variances are known and provided per point, here the nugget is
a single scalar that is optimised alongside the other hyperparameters.

Steps:
1. Load pylibkriging
2. Define the Branin function and add noise
3. Build a space-filling design and evaluate it
4. Fit a `NuggetKriging` model
5. Predict on a fine grid and plot mean + uncertainty
6. Compare with the true function
7. Inspect model parameters (including the estimated nugget)

## 0. Installation (run once)

Build and install **pylibkriging** from the local source tree.
Requires: `cmake`, a C++ compiler, Python ≥ 3.7.

A Python venv is created at the repo root to provide a consistent build
environment for cmake. The `tools/linux-macos/build.sh` script builds
the C++ core with `ENABLE_PYTHON_BINDING=on`, then `pip install` packages
the result.

In [ ]:
import sys, os
repo_root = os.path.normpath(os.path.join(os.getcwd(), '..', '..'))
venv_bin  = os.path.join(repo_root, 'bindings', 'Python', 'venv', 'bin')

# Create venv if needed
if not os.path.exists(os.path.join(venv_bin, 'python')):
    !{sys.executable} -m venv {repo_root}/bindings/Python/venv

# Install all Python build requirements into the venv
!{venv_bin}/pip install -q -r {repo_root}/bindings/Python/pylibkriging/requirements.txt \
                            -r {repo_root}/bindings/Python/pylibkriging/dev-requirements.txt

# Build the C++ core
!cd {repo_root} && \
    EXTRA_CMAKE_OPTIONS="-DPYTHON_EXECUTABLE={venv_bin}/python" \
    ENABLE_PYTHON_BINDING=on \
    bash tools/linux-macos/build.sh

In [ ]:
# Install the Python package into the current kernel
%pip install --no-build-isolation ../../bindings/Python/pylibkriging/

## 1. Load pylibkriging

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pylibkriging as lk

print("pylibkriging version:", lk.__version__)

## 2. Noisy Branin function

Same as the `NoiseKriging` example: the Branin function on $[0,1]^2$
with additive Gaussian noise ($\sigma_{\varepsilon} = 5$).

The difference is that here we **do not tell** the model the noise level.
Instead, `NuggetKriging` will estimate a single global nugget from the data.

In [ ]:
def branin(X):
    X = np.atleast_2d(X)
    x1 = X[:, 0] * 15 - 5
    x2 = X[:, 1] * 15
    return (
        (x2 - 5 / (4 * np.pi**2) * x1**2 + 5 / np.pi * x1 - 6) ** 2
        + 10 * (1 - 1 / (8 * np.pi)) * np.cos(x1)
        + 10
    )

# 50x50 evaluation grid (true function)
grid_x = np.linspace(0, 1, 50)
G1, G2  = np.meshgrid(grid_x, grid_x)
grid    = np.column_stack([G1.ravel(), G2.ravel()])
z_true  = branin(grid).reshape(50, 50)

plt.figure(figsize=(5, 4))
plt.contourf(G1, G2, z_true, levels=20, cmap='terrain')
plt.colorbar(label='Branin(x)')
plt.title('True Branin function')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 3. Design of experiments (with noise)

We sample $n = 40$ points using a Latin Hypercube Design and add
Gaussian noise with $\sigma_{\varepsilon} = 5$.

Unlike `NoiseKriging`, we do **not** pass noise variances — the model
estimates a single nugget parameter.

In [ ]:
rng = np.random.default_rng(42)

def lhs(n, d, rng):
    """Simple LHS: stratified uniform sample, independently permuted per dimension."""
    X = np.empty((n, d))
    for j in range(d):
        perm = rng.permutation(n)
        X[:, j] = (perm + rng.uniform(size=n)) / n
    return X

n = 40
noise_sd = 5.0
X = lhs(n, 2, rng)
y_true = branin(X)
y = y_true + noise_sd * rng.standard_normal(n)

plt.figure(figsize=(5, 4))
plt.contourf(G1, G2, z_true, levels=20, cmap='terrain', alpha=0.7)
sc = plt.scatter(X[:, 0], X[:, 1], c=y, cmap='terrain', edgecolors='black',
                 s=60, zorder=5, vmin=z_true.min(), vmax=z_true.max())
plt.colorbar(sc, label='Noisy Branin(x)')
plt.title(f'{n} LHS points (noise σ={noise_sd})')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

## 4. Fit a NuggetKriging model

`NuggetKriging` has the same interface as `Kriging` (no `noise` argument).
It automatically estimates the nugget effect (noise ratio) during fitting.

In [ ]:
k = lk.NuggetKriging(y, X, kernel="matern5_2", optim="BFGS10")
print(k.summary())

## 5. Predict on a fine grid

`predict()` returns `(mean, stdev, cov, mean_deriv, stdev_deriv)`.
The mean is a smooth estimate of the underlying function.

In [ ]:
pred   = k.predict(grid)
z_mean = pred[0].reshape(50, 50)
z_sd   = pred[1].reshape(50, 50)

vmin = min(z_true.min(), z_mean.min())
vmax = max(z_true.max(), z_mean.max())

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, z, title in zip(axes[:2], [z_true, z_mean], ['True Branin', 'NuggetKriging mean']):
    cf = ax.contourf(G1, G2, z, levels=20, cmap='terrain', vmin=vmin, vmax=vmax)
    ax.contour(G1, G2, z, levels=10, colors='grey', linewidths=0.5)
    ax.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=40, zorder=5)
    ax.set_title(title)
    ax.set_xlabel('$x_1$'); ax.set_ylabel('$x_2$')
    fig.colorbar(cf, ax=ax)

# Uncertainty
cf = axes[2].contourf(G1, G2, z_sd, levels=20, cmap='YlOrRd_r')
axes[2].scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=40, zorder=5)
axes[2].set_title('NuggetKriging std dev')
axes[2].set_xlabel('$x_1$'); axes[2].set_ylabel('$x_2$')
fig.colorbar(cf, ax=axes[2], label='Std dev')

plt.tight_layout()
plt.show()

## 6. Prediction error

Like `NoiseKriging`, the model smooths through the noisy data.
The estimated nugget determines the smoothing level.

In [ ]:
z_err = np.abs(z_mean - z_true)

plt.figure(figsize=(5, 4))
cf = plt.contourf(G1, G2, z_err, levels=20, cmap='Reds')
plt.scatter(X[:, 0], X[:, 1], c='white', edgecolors='black', s=40, zorder=5)
plt.colorbar(cf, label='|mean − true|')
plt.title('Absolute prediction error')
plt.xlabel('$x_1$'); plt.ylabel('$x_2$')
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((z_mean - z_true)**2))
print(f"RMSE on grid: {rmse:.4f}")

## 7. Model inspection

Key fitted parameters: length-scales $\theta$, variance $\sigma^2$,
**nugget**, and log-likelihood.

The nugget represents the estimated noise-to-signal ratio.
Compare it with the true noise variance: $\sigma_{\varepsilon}^2 = 25$.

In [ ]:
print(f"Kernel       : {k.kernel()}")
print(f"Theta (range): {np.round(k.theta(), 4)}")
print(f"Sigma2       : {k.sigma2():.4f}")
print(f"Nugget       : {k.nugget():.4f}")
print(f"LogLikelihood: {k.logLikelihood():.4f}")
print()
print(f"Estimated noise variance (nugget × sigma2): {k.nugget() * k.sigma2():.2f}")
print(f"True noise variance:                        {noise_sd**2:.2f}")

In [ ]:
# Check smoothing: predictions at training points vs observed (noisy) values
pred_train = k.predict(X)
y_pred_train = pred_train[0].ravel()

plt.figure(figsize=(5, 4))
plt.errorbar(y_true, y, yerr=2*noise_sd, fmt='o', alpha=0.5, label='Observed ± 2σ', capsize=3)
plt.scatter(y_true, y_pred_train, c='red', s=40, zorder=5, label='NuggetKriging prediction')
diag = [min(y_true.min(), y.min()), max(y_true.max(), y.max())]
plt.plot(diag, diag, 'k--', alpha=0.3)
plt.xlabel('True Branin'); plt.ylabel('Predicted / Observed')
plt.title('Smoothing effect of NuggetKriging')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Log-likelihood surface over (theta1, theta2)
theta_vals = np.linspace(0.05, 3.0, 40)
T1, T2 = np.meshgrid(theta_vals, theta_vals)
ll_mat = np.array([
    k.logLikelihoodFun(np.array([t1, t2]), return_grad=False)[0]
    for t1, t2 in zip(T1.ravel(), T2.ravel())
]).reshape(40, 40)

plt.figure(figsize=(5, 4))
plt.contourf(T1, T2, ll_mat, levels=20, cmap='Blues')
plt.colorbar(label='Log-likelihood')
theta_hat = k.theta()
plt.plot(theta_hat[0], theta_hat[1], 'r+', markersize=12, markeredgewidth=2, label='optimum')
plt.legend()
plt.title('Log-likelihood surface')
plt.xlabel(r'$\theta_1$'); plt.ylabel(r'$\theta_2$')
plt.tight_layout()
plt.show()